In [1]:
from itertools import product


# Logical implication: P -> Q
def implies(P, Q):
    return (not P) or Q


# Knowledge Base:
# KB = (Q -> P) AND (P -> NOT Q) AND (Q OR R)
def evaluate_kb(P, Q, R):
    return (
        implies(Q, P)
        and implies(P, not Q)
        and (Q or R)
    )


# Evaluate the query
# 1: R
# 2: R -> P
# 3: Q -> R
def evaluate_query(choice, P, Q, R):
    if choice == 1:
        return R
    elif choice == 2:
        return implies(R, P)
    elif choice == 3:
        return implies(Q, R)


# Truth-table enumeration algorithm
def entails(choice):
    # Generate all 2^3 = 8 truth assignments
    for P, Q, R in product([False, True], repeat=3):

        kb = evaluate_kb(P, Q, R)
        query = evaluate_query(choice, P, Q, R)

        # Counterexample: KB is true, but query is false
        if kb and not query:
            return False

    # No counterexample found
    return True


# Display the complete truth table
def print_truth_table():
    print("TRUTH TABLE ENUMERATION\n")

    print("P Q R | Q->P P->!Q Q|R | KB")
    print("--------------------------------")

    for P, Q, R in product([False, True], repeat=3):

        first = implies(Q, P)
        second = implies(P, not Q)
        third = Q or R
        kb = evaluate_kb(P, Q, R)

        # Convert Boolean values to T/F
        tf = lambda x: "T" if x else "F"

        print(
            f"{tf(P)} {tf(Q)} {tf(R)} |"
            f"   {tf(first)}    {tf(second)}    "
            f"{tf(third)} | {tf(kb)}"
        )


# Main program
print_truth_table()

print("\nENTAILMENT RESULTS")

queries = {
    1: "R",
    2: "(R -> P)",
    3: "(Q -> R)"
}

for choice, query in queries.items():
    result = entails(choice)
    answer = "YES" if result else "NO"

    print(f"KB entails {query}: {answer}")


TRUTH TABLE ENUMERATION

P Q R | Q->P P->!Q Q|R | KB
--------------------------------
F F F |   T    T    F | F
F F T |   T    T    T | T
F T F |   F    T    T | F
F T T |   F    T    T | F
T F F |   T    T    F | F
T F T |   T    T    T | T
T T F |   T    F    T | F
T T T |   T    F    T | F

ENTAILMENT RESULTS
KB entails R: YES
KB entails (R -> P): NO
KB entails (Q -> R): YES


In [4]:
def unify(x, y, subst=None):
    if subst is None:
        subst = {}

    # Apply existing substitutions
    while isinstance(x, str) and x in subst:
        x = subst[x]

    while isinstance(y, str) and y in subst:
        y = subst[y]

    # Step 1: If both expressions are identical
    if x == y:
        return subst

    # Step 2: If x is a variable
    if isinstance(x, str) and x.islower():
        if occurs(x, y, subst):
            return None
        subst[x] = y
        return subst

    # Step 3: If y is a variable
    if isinstance(y, str) and y.islower():
        if occurs(y, x, subst):
            return None
        subst[y] = x
        return subst

    # Step 4: Both must be lists with the same predicate
    if isinstance(x, list) and isinstance(y, list):
        if len(x) != len(y) or x[0] != y[0]:
            return None

        # Step 5: Unify corresponding arguments
        for a, b in zip(x[1:], y[1:]):
            subst = unify(a, b, subst)
            if subst is None:
                return None
        return subst

    # Step 6: Unification failed
    return None


def occurs(var, expr, subst):
    if isinstance(expr, str):
        while expr in subst:
            expr = subst[expr]
        return expr == var

    if isinstance(expr, list):
        return any(occurs(var, item, subst) for item in expr)

    return False


# Convert user input into a nested list expression
def parse_expr(expr):
    expr = expr.replace(" ", "")

    # If expression is a simple variable or constant
    if "(" not in expr:
        return expr

    # Extract predicate/function name
    name = expr[:expr.index("(")]

    # Extract arguments
    args = expr[expr.index("(") + 1:-1]

    result = [name]

    # Parse each argument
    for arg in args.split(","):
        result.append(parse_expr(arg))

    return result


# User input
expr1 = input("Enter first expression: ")
expr2 = input("Enter second expression: ")

# Convert input expressions
expr1 = parse_expr(expr1)
expr2 = parse_expr(expr2)

# Perform unification
result = unify(expr1, expr2)

# Display result
if result is not None:
    print("Unification Successful!")
    print("Substitution Set:", result)
else:
    print("Unification Failed!")


Enter first expression: P(x,f(x),y)
Enter second expression: P(a,f(a),b)
Unification Successful!
Substitution Set: {'x': 'a', 'y': 'b'}
